# 🏥 Apollo Hospital Voice AI Assistant (Edge TTS Version)

**Fast & Free Real-time Voice AI for Indian Languages**

This notebook implements a complete Voice AI pipeline using **Edge TTS** for high-quality, fast, and free text-to-speech in Indian languages.

### 🌍 Supported Languages:
- 🇮🇳 **Kannada** (kn)
- 🇮🇳 **Tamil** (ta)
- 🇮🇳 **Telugu** (te)
- 🇮🇳 **Hindi** (hi)
- 🌏 **English** (en)

### 🚀 Pipeline Stages:
1. **VAD** - Silero Voice Activity Detection
2. **STT** - Faster-Whisper (Large-v3)
3. **Language Detection** - Unicode Script Analysis
4. **Signal Extraction** - Intent & Urgency
5. **Safety Gate** - Healthcare Safety Checks
6. **Policy Engine** - Hospital Rules
7. **LLM** - LLaMA 3.1-8B (4-bit)
8. **TTS** - **Microsoft Edge TTS** (Neural Voices)

## 📦 Step 1: Install Dependencies

In [ ]:
# Install core AI packages
!pip install -q torch torchaudio
!pip install -q faster-whisper
!pip install -q transformers accelerate bitsandbytes
!pip install -q soundfile librosa

# Install Edge TTS and utilities
!pip install -q edge-tts
!pip install -q nest_asyncio
!pip install -q ipywidgets

print("✅ All dependencies installed!")

## 🔧 Step 2: Device & Language Setup

In [ ]:
import os
import io
import time
import asyncio
import tempfile
import numpy as np
import torch
import soundfile as sf
import edge_tts
import nest_asyncio
from IPython.display import display, Audio, HTML, clear_output
import ipywidgets as widgets

# Apply nested asyncio to allow running async loops in Jupyter
nest_asyncio.apply()

# Device Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Hardware: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

# Set Hugging Face Token (for LLaMA)
HF_TOKEN = os.environ.get("HUGGINGFACE_TOKEN", os.environ.get("HF_TOKEN", ""))
if not HF_TOKEN:
    print("⚠️  NOTE: You need a Hugging Face token for LLaMA access!")
    print("   Set it via: os.environ['HF_TOKEN'] = 'your_token'")

# ============================================================================
# 🗣️ EDGE TTS VOICE CONFIGURATION (CRITICAL)
# ============================================================================

SUPPORTED_LANGUAGES = {
    'kn': 'Kannada',
    'ta': 'Tamil', 
    'te': 'Telugu',
    'hi': 'Hindi',
    'en': 'English'
}

# High-quality Neural voices for Indian languages
EDGE_TTS_VOICES = {
    'kn': 'kn-IN-SapnaNeural',      # Kannada female
    'ta': 'ta-IN-PallaviNeural',    # Tamil female
    'te': 'te-IN-ShrutiNeural',     # Telugu female
    'hi': 'hi-IN-SwaraNeural',      # Hindi female
    'en': 'en-IN-NeerjaNeural',     # English (India) female
}

print("✅ Edge TTS Voices Configured:")
for lang, voice in EDGE_TTS_VOICES.items():
    print(f"   - {SUPPORTED_LANGUAGES[lang]}: {voice}")

## 📚 Step 3: Hospital Knowledge & Config

In [ ]:
# Hospital Configuration
HOSPITAL_CONFIG = {
    "hospital_name": "Apollo Hospital",
    "city": "Bengaluru",
    "emergency_number": "108",
    "helpline": "1860-500-1066",
    "disclaimer_text": "This is AI advice. Please consult a real doctor.",
}

# Unicode ranges for language detection
SCRIPT_RANGES = {
    'kn': [(0x0C80, 0x0CFF)],  # Kannada
    'ta': [(0x0B80, 0x0BFF)],  # Tamil
    'te': [(0x0C00, 0x0C7F)],  # Telugu
    'hi': [(0x0900, 0x097F)],  # Hindi
    'en': [(0x0041, 0x005A), (0x0061, 0x007A)],  # English
}

# Key Signals
INTENT_KEYWORDS = {
    'medical_query': {
        'en': ['pain', 'fever', 'headache', 'cough', 'cold', 'stomach', 'doctor', 'medicine', 'treatment', 'symptom'],
        'hi': ['दर्द', 'बुखार', 'सिरदर्द', 'खांसी', 'सर्दी', 'पेट', 'डॉक्टर', 'दवाई', 'इलाज'],
        'ta': ['வலி', 'காய்ச்சல்', 'தலைவலி', 'இருமல்', 'சளி', 'வயிறு', 'மருத்துவர்'],
        'te': ['నొప్పి', 'జ్వరం', 'తలనొప్పి', 'దగ్గు', 'జలుబు', 'కడుపు', 'డాక్టర్'],
        'kn': ['ನೋವು', 'ಜ್ವರ', 'ತಲೆನೋವು', 'ಕೆಮ್ಮು', 'ಶೀತ', 'ಹೊಟ್ಟೆ', 'ವೈದ್ಯರು'],
    }
}

CRITICAL_SYMPTOMS = {
    'en': ['chest pain', 'heart attack', "can't breathe", 'unconscious', 'bleeding'],
    'hi': ['छाती में दर्द', 'हार्ट अटैक', 'सांस नहीं', 'बेहोश', 'खून'],
    'ta': ['நெஞ்சு வலி', 'மாரடைப்பு', 'மூச்சு திணறல்', 'மயக்கம்', 'இரத்தம்'],
    'te': ['ఛాతీ నొప్పి', 'గుండెపోటు', 'ఊపిరి ఆడటం లేదు', 'స్పృహ లేదు', 'రక్తం'],
    'kn': ['ಎದೆ ನೋವು', 'ಹೃದಯಾಘಾತ', 'ಉಸಿರಾಟ ತೊಂದರೆ', 'ಪ್ರಜ್ಞೆ ತಪ್ಪು', 'ರಕ್ತ'],
}

print("✅ Hospital knowledge base loaded!")

## 🤖 Step 4: Load AI Models

In [ ]:
# Model Container
class ModelManager:
    def __init__(self):
        self.vad_model = None
        self.vad_utils = None
        self.whisper_model = None
        self.llm_model = None
        self.llm_tokenizer = None
        self.loaded = False

models = ModelManager()

def load_models():
    print("⏳ Loading models... this may take 3-5 minutes...")
    start_total = time.time()
    
    # 1. Load VAD
    print("   1/3 Loading Silero VAD...", end=" ")
    models.vad_model, models.vad_utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad', onnx=False)
    models.vad_model.to(DEVICE)
    print("✅")
    
    # 2. Load Whisper STT (Large-v3)
    print("   2/3 Loading Faster-Whisper (large-v3)...", end=" ")
    from faster_whisper import WhisperModel
    compute_type = "float16" if DEVICE == "cuda" else "int8"
    models.whisper_model = WhisperModel("large-v3", device=DEVICE, compute_type=compute_type)
    print("✅")
    
    # 3. Load LLaMA LLM
    print("   3/3 Loading LLaMA 3.1-8B (4-bit)...", end=" ")
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    
    model_id = "meta-llama/Llama-3.1-8B-Instruct"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    models.llm_tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    models.llm_model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        quantization_config=bnb_config, 
        device_map="auto", 
        token=HF_TOKEN
    )
    print("✅")
    
    models.loaded = True
    print(f"\n🎉 All models loaded successfully in {time.time() - start_total:.1f}s!")
    print("NOTE: TTS does not need loading (Edge TTS is cloud-based)")

# Run loading
load_models()

## ⚙️ Step 5: The Full Pipeline

In [ ]:
# ====================== PIPELINE FUNCTIONS ======================

# 1. VAD
def run_vad(audio_in, sr):
    if sr != 16000:
        # Simple resample implementation wrapper if needed
        import librosa
        audio_in = librosa.resample(audio_in, orig_sr=sr, target_sr=16000)
        sr = 16000
    tensor = torch.FloatTensor(audio_in).to(DEVICE)
    get_speech_ts = models.vad_utils[0]
    timestamps = get_speech_ts(tensor, models.vad_model, sampling_rate=sr)
    return len(timestamps) > 0

# 2. STT
def run_stt(audio_in, sr, lang=None):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        sf.write(f.name, audio_in, sr)
        temp_path = f.name
    try:
        segments, info = models.whisper_model.transcribe(temp_path, language=lang, beam_size=5)
        text = " ".join([s.text for s in segments]).strip()
        return text, info.language
    finally:
        os.unlink(temp_path)

# 3. Language Detect
def detect_lang(text):
    counts = {k: 0 for k in SCRIPT_RANGES}
    for char in text:
        cp = ord(char)
        for lang, ranges in SCRIPT_RANGES.items():
            for start, end in ranges:
                if start <= cp <= end:
                    counts[lang] += 1
    if sum(counts.values()) == 0: return 'en'
    return max(counts, key=counts.get)

# 4. Safety
def safety_check(text, lang):
    escalate = False
    reason = ""
    keywords = CRITICAL_SYMPTOMS.get(lang, []) + CRITICAL_SYMPTOMS.get('en', [])
    for kw in keywords:
        if kw in text.lower():
            escalate = True
            reason = f"Critical symptom: {kw}"
            break
    return escalate, reason

# 5. LLM Response
def generate_response(text, lang, escalate):
    lang_name = SUPPORTED_LANGUAGES.get(lang, 'English')
    system = f"""You are Apollo Hospital's AI assistant.
    Rules:
    1. Respond in {lang_name} (native script).
    2. Be concise (max 50 words).
    3. Tone: Professional & Empathetic.
    4. {'URGENT: EMERGENCY DETECTED. Recommend doctor immediately!' if escalate else 'If symptoms persist, see a doctor.'}
    """
    
    messages = [{"role": "system", "content": system}, {"role": "user", "content": text}]
    prompt = models.llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = models.llm_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        out = models.llm_model.generate(**inputs, max_new_tokens=150, temperature=0.7)
    return models.llm_tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# 6. TTS (Edge TTS)
async def generate_tts_async(text, lang):
    voice = EDGE_TTS_VOICES.get(lang, 'en-IN-NeerjaNeural')
    with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
        path = f.name
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(path)
    audio, sr = sf.read(path)
    os.unlink(path)
    return audio, sr

def generate_tts(text, lang):
    return asyncio.get_event_loop().run_until_complete(generate_tts_async(text, lang))

# ====== MAIN PROCESSING FUNCTION ======

def process_audio(file_path, forced_lang=None):
    print("Working...", end="\r")
    start = time.time()
    
    # Load Audio
    audio, sr = sf.read(file_path)
    if len(audio.shape) > 1: audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    
    # 1. VAD
    if not run_vad(audio, sr):
        return {"error": "No speech detected!"}
        
    # 2. STT
    text, whisper_lang = run_stt(audio, sr, lang=forced_lang)
    
    # 3. Detect Lang
    if forced_lang:
        lang = forced_lang
    else:
        lang = detect_lang(text)
        if lang == 'en' and whisper_lang in SUPPORTED_LANGUAGES:
            lang = whisper_lang 
        
    # 4. Safety
    escalate, reason = safety_check(text, lang)
    
    # 5. LLM
    response = generate_response(text, lang, escalate)
    
    # 6. TTS
    audio_out, sr_out = generate_tts(response, lang)
    
    latency = (time.time() - start) * 1000
    
    return {
        "success": True,
        "input_text": text,
        "language": SUPPORTED_LANGUAGES.get(lang, lang),
        "escalation": reason if escalate else "None",
        "response_text": response,
        "audio": (audio_out, sr_out),
        "latency": f"{latency:.0f}ms"
    }

print("✅ Pipeline Ready!")

## 🎤 Step 6: Test Interface

Select a language (or leave generic) and upload an audio file.

In [ ]:
# Widgets
uploader = widgets.FileUpload(accept='.wav,.mp3', description="Upload Audio")

lang_dropdown = widgets.Dropdown(
    options=[('Auto-detect', None)] + [(v, k) for k, v in SUPPORTED_LANGUAGES.items()],
    value=None,
    description='Language:',
    style={'description_width': 'initial'}
)

btn = widgets.Button(description="🎯 Process Audio", button_style='success', layout=widgets.Layout(width='200px'))
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        if not uploader.value: 
            print("⚠️ Please upload a file first.")
            return
            
        # Handle file upload structure (ipywidgets compatibility)
        try:
            if isinstance(uploader.value, tuple):
                file_info = uploader.value[0]
            elif isinstance(uploader.value, dict):
                file_info = list(uploader.value.values())[0]
            else:
                file_info = uploader.value[0]
                
            content = file_info.get('content') or file_info.get('data') 
            fname = file_info.get('name', 'audio.wav')
        except Exception as e:
            print(f"⚠️ Error reading widget: {e}")
            return
        
        print(f"📂 Processing: {fname}...")

        # Save temp
        suffix = os.path.splitext(fname)[1] or ".wav"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
            f.write(content)
            path = f.name
            
        try:
            # Process with selected language
            forced = lang_dropdown.value
            res = process_audio(path, forced_lang=forced)
            
            if "error" in res:
                print(f"❌ Error: {res['error']}")
            else:
                print("\n" + "="*40)
                print(f"⏱️ Latency: {res['latency']}")
                print(f"🌍 Language: {res['language']}")
                print("="*40)
                print(f"🗣️ You: \"{res['input_text']}\"")
                print("-"*40)
                if res['escalation'] != "None":
                    print(f"🚨 ALERT: {res['escalation']}")
                print(f"🤖 Apollo: \"{res['response_text']}\"")
                print("="*40)
                
                # Play Audio
                print("🔊 Playing Response...")
                audio_data, sr = res['audio']
                display(Audio(audio_data, rate=sr, autoplay=True))
                
        except Exception as e:
            print(f"❌ Execution Error: {e}")
            import traceback
            traceback.print_exc()
        finally:
            if os.path.exists(path): os.unlink(path)

btn.on_click(on_click)

# Display UI
display(widgets.HTML("<h3>🎤 Apollo Voice AI Demo</h3>"))
display(widgets.HBox([uploader, lang_dropdown]))
display(btn)
display(out)